In [3]:
import os, glob
from petsc4py import PETSc
from scipy.io import mmread

/home/alexandr/Plocha/bsc_thesis/bsc_venv/lib/python3.12/site-packages/petsc4py/lib/__init__.py:44: UserWarning: ignored arch: 'arch-linux-c-debug', using: './arch-linux-c-opt/'
  path, arch = getPathArch(path, arch, rcvar, rcfile)


### Convert matrices from .mtx to .bin

In [4]:
def mtx_to_bin_mat(input_file: str, output_file: str):
    # load Matrix Market file
    A = mmread(input_file).tocsr()

    # create PETSc matrix (AIJ = compressed sparse row)
    petsc_A = PETSc.Mat().createAIJ(size=A.shape,
                                    csr=(A.indptr, A.indices, A.data))

    # write to PETSc binary format
    viewer = PETSc.Viewer().createBinary(output_file, PETSc.Viewer.Mode.WRITE)
    petsc_A.view(viewer)
    viewer.destroy()
    petsc_A.destroy()

In [5]:
mat_input_path  = "./mtx/"
mat_output_path = "./bin/"

os.makedirs(mat_output_path, exist_ok=True)

for f in glob.glob(os.path.join(mat_input_path, "*.mtx")):
    name = os.path.splitext(os.path.basename(f))[0]
    out_file = os.path.join(mat_output_path, name + ".bin")
    print(f"Converting {name}.mtx -> {name}.bin")
    mtx_to_bin_mat(f, out_file)

Converting tandem_dual.mtx -> tandem_dual.bin


### Convert vectors from .mtx to .bin

In [6]:
def mtx_to_bin_vec(input_file: str, output_file: str):
    # load Matrix Market file (as dense 2D array)
    v = mmread(input_file)

    # make sure it’s a flat 1D vector
    v = v.reshape(-1)

    # create PETSc vector and set values
    petsc_v = PETSc.Vec().createSeq(len(v))
    petsc_v.setArray(v)

    # write to PETSc binary format
    viewer = PETSc.Viewer().createBinary(output_file, PETSc.Viewer.Mode.WRITE)
    petsc_v.view(viewer)
    viewer.destroy()
    petsc_v.destroy()

In [7]:
vec_input_path  = "./mtx/vec/"
vec_output_path = "./bin/vec/"

os.makedirs(vec_output_path, exist_ok=True)

for f in glob.glob(os.path.join(vec_input_path, "*.mtx")):
    name = os.path.splitext(os.path.basename(f))[0]
    out_file = os.path.join(vec_output_path, name + ".bin")
    print(f"Converting {name}.mtx -> {name}.bin")
    mtx_to_bin_vec(f, out_file)